# Exhaustive Per-Shot Correlation Analysis: Logical Error Distribution vs Fixed-Class LLRs

This notebook performs **exhaustive correlation analysis** by exploring ALL logical classes (2^k)
for each shot. This provides complete data for analyzing whether the logical error distribution
predicts which error class will have the lowest fixed-class LLR.

Key differences from sampled analysis:
- **Complete coverage**: Explores all 2^k logical classes, not just sampled representatives
- **Parallelization**: Uses `num_procs_for_gap` for efficient parallel decoding
- **Fewer shots**: Due to computational cost, typically 10 shots vs 1000+ in sampled mode

Key metrics:
1. **Per-shot Spearman correlation**: Does rank predict LLR ordering within each shot?
2. **Optimal selection rate**: How often does the highest-probability error have the lowest LLR?
3. **Complete top-k success curve**: P(winner in top-k) for all k values
4. **Winning rank distribution**: Which ranks win most often?

## 1. Data Collection Configuration

Configure the simulation parameters below, then run the cell to generate the script command.

In [1]:
# ==============================================================================
# CONFIGURATION - Set your simulation parameters here
# ==============================================================================

# Code parameters
N_QUBITS = 144          # BB code variant: 72, 90, 108, 144, 288, 360, 756
P = 0.003               # Physical error rate

# Simulation parameters (exhaustive mode)
N_SHOTS = 10            # Number of shots to simulate (keep small, exhaustive is expensive)
NUM_PROCS_FOR_GAP = 126 # Parallel processes for gap computation
SEED = 42               # Random seed

# Output path (auto-generated based on parameters)
OUTPUT_DIR = "simulations/data/correlation_analysis"

In [2]:
# ==============================================================================
# Generate script command based on configuration
# ==============================================================================

from pathlib import Path

# Generate output filename
OUTPUT_FILENAME = f"bb_n{N_QUBITS}_p{P}_exhaustive_shots{N_SHOTS}.parquet"
OUTPUT_PATH = Path(OUTPUT_DIR) / OUTPUT_FILENAME

# Build command
SCRIPT_PATH = "simulations/analysis/scripts/run_distribution_correlation_analysis.py"

cmd_parts = [
    f"python {SCRIPT_PATH}",
    f"    --n-qubits {N_QUBITS}",
    f"    --p {P}",
    f"    --n-shots {N_SHOTS}",
    f"    --exhaustive",
    f"    --num-procs-for-gap {NUM_PROCS_FOR_GAP}",
    f"    --seed {SEED}",
    f"    --output {OUTPUT_PATH}",
]

command = " \\\n".join(cmd_parts)

# Display
print("=" * 70)
print("GENERATED COMMAND (Exhaustive Mode)")
print("=" * 70)
print()
print(command)
print()
print("=" * 70)
print(f"Output file: {OUTPUT_PATH}")
print("=" * 70)

# Store for later use
RESULTS_PATH = Path("../../..") / OUTPUT_PATH

GENERATED COMMAND (Exhaustive Mode)

python simulations/analysis/scripts/run_distribution_correlation_analysis.py \
    --n-qubits 144 \
    --p 0.003 \
    --n-shots 10 \
    --exhaustive \
    --num-procs-for-gap 126 \
    --seed 42 \
    --output simulations/data/correlation_analysis/bb_n144_p0.003_exhaustive_shots10.parquet

Output file: simulations/data/correlation_analysis/bb_n144_p0.003_exhaustive_shots10.parquet


In [ ]:
# Autoreload extension for Jupyter notebooks
%load_ext autoreload
%autoreload 2

# Standard library imports
import json
from pathlib import Path

# Third-party library imports
import numpy as np
import pandas as pd
from scipy import stats

# Plotly for interactive visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.6f}'.format)

## 2. Load Results

In [ ]:
# ==============================================================================
# Load results (uses RESULTS_PATH from configuration above)
# ==============================================================================

if not RESULTS_PATH.exists():
    print(f"Results file not found: {RESULTS_PATH.resolve()}")
    print("\nPlease run the data collection script first using the command above.")
    raise FileNotFoundError(f"Results not found at {RESULTS_PATH}")

# Load DataFrame
df_results = pd.read_parquet(RESULTS_PATH)

# Load metadata
metadata_path = RESULTS_PATH.with_suffix(".json")
if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)
else:
    metadata = {}

# Display summary
print(f"Loaded {len(df_results):,} records from {RESULTS_PATH.name}")
print(f"\nMetadata:")
for key, value in metadata.items():
    if isinstance(value, dict):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value}")

print(f"\nDataFrame preview:")
df_results.head(10)

In [ ]:
# ==============================================================================
# Data Summary
# ==============================================================================

n_shots = df_results['shot_id'].nunique()
n_errors_per_shot = len(df_results) // n_shots
total_logical_classes = metadata.get('total_logical_classes', n_errors_per_shot + 1)

print("=" * 60)
print("DATA SUMMARY")
print("=" * 60)
print(f"\nTotal shots: {n_shots}")
print(f"Errors per shot: {n_errors_per_shot}")
print(f"Total logical classes: {total_logical_classes}")
print(f"Total records: {len(df_results):,}")
print(f"\nError rank range: {df_results['error_rank'].min()} to {df_results['error_rank'].max()}")
print(f"LLR range: {df_results['fixed_llr'].min():.2f} to {df_results['fixed_llr'].max():.2f}")

## 3. Per-Shot Correlation Analysis

In [ ]:
# ==============================================================================
# Compute Per-Shot Statistics
# ==============================================================================

per_shot_stats = []

for shot_id, group in df_results.groupby("shot_id"):
    # Sort by error rank to ensure consistent ordering
    group = group.sort_values("error_rank")
    
    # Per-shot Spearman correlation (rank vs LLR)
    rho, p_val = stats.spearmanr(group["error_rank"], group["fixed_llr"])
    
    # Find which rank achieved the minimum LLR
    min_idx = group["fixed_llr"].idxmin()
    min_llr_rank = group.loc[min_idx, "error_rank"]
    min_llr = group["fixed_llr"].min()
    
    # Rank-0 statistics (most likely error)
    rank0_llr = group[group["error_rank"] == 0]["fixed_llr"].values[0] if 0 in group["error_rank"].values else None
    rank0_is_best = (min_llr_rank == 0)
    
    # Regret: LLR penalty for using rank-0 instead of optimal
    regret = (rank0_llr - min_llr) if rank0_llr is not None else None
    
    # Baseline difficulty (best_llr from standard decoding)
    best_llr = group["best_llr"].iloc[0]
    
    # Mean LLR for random selection baseline
    mean_llr = group["fixed_llr"].mean()
    random_regret = mean_llr - min_llr
    
    per_shot_stats.append({
        "shot_id": shot_id,
        "spearman_rho": rho,
        "spearman_p": p_val,
        "min_llr_rank": min_llr_rank,
        "min_llr": min_llr,
        "rank0_llr": rank0_llr,
        "rank0_is_best": rank0_is_best,
        "regret": regret,
        "random_regret": random_regret,
        "best_llr": best_llr,
        "mean_fixed_llr": mean_llr,
    })

df_per_shot = pd.DataFrame(per_shot_stats)

print("=" * 60)
print("PER-SHOT STATISTICS COMPUTED")
print("=" * 60)
print(f"\nAnalyzed {len(df_per_shot)} shots")
print(f"Errors per shot: {n_errors_per_shot}")
print(f"\nDataFrame preview:")
df_per_shot

## 4. Per-Shot Spearman Correlation Statistics

In [ ]:
# ==============================================================================
# Per-Shot Correlation Statistics
# ==============================================================================

print("=" * 60)
print("PER-SHOT SPEARMAN CORRELATION STATISTICS")
print("=" * 60)

# Filter out NaN correlations (can happen if all LLRs are identical in a shot)
valid_correlations = df_per_shot["spearman_rho"].dropna()
n_valid = len(valid_correlations)
n_total = len(df_per_shot)

print(f"\nValid correlations: {n_valid}/{n_total} shots")

# Basic statistics
mean_rho = valid_correlations.mean()
std_rho = valid_correlations.std()
median_rho = valid_correlations.median()

print(f"\nPer-shot Spearman correlation (rank vs LLR):")
print(f"  Mean:   {mean_rho:.4f}")
print(f"  Std:    {std_rho:.4f}")
print(f"  Median: {median_rho:.4f}")
print(f"  Min:    {valid_correlations.min():.4f}")
print(f"  Max:    {valid_correlations.max():.4f}")

# One-sample t-test: is mean correlation significantly different from 0?
if n_valid > 1:
    t_stat, t_pval = stats.ttest_1samp(valid_correlations, 0)
    print(f"\nOne-sample t-test (H0: mean = 0):")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {t_pval:.2e}")
    print(f"  Interpretation: {'Mean is significantly different from 0' if t_pval < 0.05 else 'Mean is NOT significantly different from 0'}")
else:
    t_pval = None
    print("\nInsufficient samples for t-test")

# Fraction of positive/negative correlations
n_positive = (valid_correlations > 0).sum()
n_negative = (valid_correlations < 0).sum()
n_zero = (valid_correlations == 0).sum()
print(f"\nCorrelation sign distribution:")
print(f"  Positive (rank ↑ → LLR ↑): {n_positive} ({100*n_positive/n_valid:.1f}%)")
print(f"  Negative (rank ↑ → LLR ↓): {n_negative} ({100*n_negative/n_valid:.1f}%)")
print(f"  Zero:                       {n_zero} ({100*n_zero/n_valid:.1f}%)")

In [ ]:
# ==============================================================================
# Histogram of Per-Shot Correlations
# ==============================================================================

fig = go.Figure()

# Histogram of per-shot Spearman correlations
fig.add_trace(go.Histogram(
    x=valid_correlations,
    nbinsx=max(10, n_valid // 2),
    name="Per-shot ρ",
    marker_color="steelblue",
    opacity=0.7,
))

# Add vertical line at mean
fig.add_vline(
    x=mean_rho,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean = {mean_rho:.3f}",
    annotation_position="top",
)

# Add vertical line at 0
fig.add_vline(
    x=0,
    line_dash="solid",
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title="Distribution of Per-Shot Spearman Correlations (Rank vs LLR) - EXHAUSTIVE",
    xaxis_title="Spearman ρ (per shot)",
    yaxis_title="Count",
    showlegend=False,
)

# Add annotation with statistics
stats_text = f"Mean = {mean_rho:.3f}<br>Std = {std_rho:.3f}"
if t_pval is not None:
    stats_text += f"<br>t-test p = {t_pval:.2e}"
fig.add_annotation(
    text=stats_text,
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    showarrow=False,
    font=dict(size=11),
    bgcolor="white",
    bordercolor="gray",
    borderwidth=1,
    align="left",
)

fig.show()

## 5. Optimal Selection Rate & Regret Analysis

In [ ]:
# ==============================================================================
# Optimal Selection Rate and Regret Analysis
# ==============================================================================

print("=" * 60)
print("OPTIMAL SELECTION RATE")
print("=" * 60)

random_baseline = 1.0 / n_errors_per_shot

# Rank-0 selection success rate
rank0_success_rate = df_per_shot["rank0_is_best"].mean()
print(f"\nIf we always select rank-0 (highest probability) error:")
print(f"  Success rate (rank-0 has min LLR): {100*rank0_success_rate:.2f}%")
print(f"  Random baseline:                   {100*random_baseline:.4f}%")
print(f"  Improvement over random:           {100*(rank0_success_rate - random_baseline):.2f}pp")
print(f"  Improvement factor:                {rank0_success_rate / random_baseline:.1f}x")

# Binomial test: is success rate better than random?
if n_shots > 1:
    binom_result = stats.binomtest(
        k=int(df_per_shot["rank0_is_best"].sum()),
        n=len(df_per_shot),
        p=random_baseline,
        alternative="greater",
    )
    print(f"\nBinomial test (H0: success rate = random):")
    print(f"  p-value: {binom_result.pvalue:.2e}")
    print(f"  95% CI: [{binom_result.proportion_ci(confidence_level=0.95).low:.4f}, {binom_result.proportion_ci(confidence_level=0.95).high:.4f}]")

print("\n" + "=" * 60)
print("REGRET ANALYSIS")
print("=" * 60)

valid_regrets = df_per_shot["regret"].dropna()
mean_regret = valid_regrets.mean()
mean_random_regret = df_per_shot["random_regret"].mean()

print(f"\nRegret = LLR(selected) - LLR(optimal)")
print(f"\nRank-0 selection regret:")
print(f"  Mean:   {mean_regret:.2f}")
print(f"  Median: {valid_regrets.median():.2f}")
print(f"  Std:    {valid_regrets.std():.2f}")

print(f"\nRandom selection regret (expected):")
print(f"  Mean:   {mean_random_regret:.2f}")
print(f"  Median: {df_per_shot['random_regret'].median():.2f}")
print(f"  Std:    {df_per_shot['random_regret'].std():.2f}")

print(f"\nRegret reduction (random → rank-0): {mean_random_regret - mean_regret:.2f}")

## 6. Complete Top-k Success Rate Curve

In [ ]:
# ==============================================================================
# Complete Top-k Success Rate (Exhaustive Analysis)
# ==============================================================================

print("=" * 60)
print("COMPLETE TOP-K SUCCESS RATE")
print("=" * 60)
print("\nIf we pick among top-k errors by distribution probability,")
print("how often does the true minimum LLR fall within that set?")
print()

unique_ranks = sorted(df_results["error_rank"].unique())
n_total_ranks = len(unique_ranks)

topk_results = []
for k in range(1, min(n_total_ranks + 1, 101)):  # Cap at 100 for display
    top_k_ranks = set(unique_ranks[:k])
    success_count = (df_per_shot["min_llr_rank"].isin(top_k_ranks)).sum()
    success_rate = success_count / len(df_per_shot)
    random_rate = k / n_total_ranks
    topk_results.append({
        "k": k,
        "success_rate": success_rate,
        "random_rate": random_rate,
        "improvement": success_rate - random_rate,
    })

df_topk = pd.DataFrame(topk_results)

# Show summary (first and last 10)
print("First 10 k values:")
print(df_topk.head(10).to_string(index=False, formatters={
    "success_rate": "{:.2%}".format,
    "random_rate": "{:.2%}".format,
    "improvement": "{:+.2%}".format,
}))

if len(df_topk) > 20:
    print("\n...\n")
    print("Last 10 k values:")
    print(df_topk.tail(10).to_string(index=False, formatters={
        "success_rate": "{:.2%}".format,
        "random_rate": "{:.2%}".format,
        "improvement": "{:+.2%}".format,
    }))

In [ ]:
# ==============================================================================
# Visualization: Complete Top-k Success Rate
# ==============================================================================

fig = go.Figure()

# Actual success rate
fig.add_trace(go.Scatter(
    x=df_topk["k"],
    y=df_topk["success_rate"],
    mode="lines+markers" if len(df_topk) <= 50 else "lines",
    name="Actual (distribution-guided)",
    line=dict(color="steelblue", width=2),
    marker=dict(size=6),
))

# Random baseline
fig.add_trace(go.Scatter(
    x=df_topk["k"],
    y=df_topk["random_rate"],
    mode="lines",
    name="Random baseline",
    line=dict(color="red", width=2, dash="dash"),
))

fig.update_layout(
    title="Complete Top-k Success Rate: Does Distribution Rank Predict Winner? (EXHAUSTIVE)",
    xaxis_title="k (number of top ranks considered)",
    yaxis_title="Success Rate (min LLR in top-k)",
    yaxis_tickformat=".0%",
    legend=dict(x=0.6, y=0.1),
    xaxis_type="log" if n_total_ranks > 100 else "linear",
)

fig.show()

## 7. Distribution of Winning Rank

In [ ]:
# ==============================================================================
# Distribution of Winning Rank
# ==============================================================================

print("=" * 60)
print("DISTRIBUTION OF WINNING RANK")
print("=" * 60)
print("\nWhich rank achieves the minimum LLR most often?")
print()

win_counts = df_per_shot["min_llr_rank"].value_counts().sort_index()
expected_pct = 100.0 / n_total_ranks

# Show top 20 winning ranks
top_winners = win_counts.head(20)

winner_df = pd.DataFrame({
    "rank": top_winners.index,
    "wins": top_winners.values,
    "win_pct": top_winners.values / len(df_per_shot) * 100,
    "expected_pct": expected_pct,
})
winner_df["vs_expected"] = winner_df["win_pct"] - expected_pct

print(f"Top 20 winning ranks (expected: {expected_pct:.4f}% each):")
print(winner_df.to_string(index=False, formatters={
    "win_pct": "{:.2f}%".format,
    "expected_pct": "{:.4f}%".format,
    "vs_expected": "{:+.2f}pp".format,
}))

print(f"\nTotal unique winning ranks: {len(win_counts)} / {n_total_ranks}")

In [ ]:
# ==============================================================================
# Visualization: Winning Rank Distribution
# ==============================================================================

fig = go.Figure()

# Bar chart of win percentages (top 50 or all if fewer)
n_display = min(50, len(win_counts))
top_winners = win_counts.head(n_display)

fig.add_trace(go.Bar(
    x=[str(r) for r in top_winners.index],
    y=top_winners.values / len(df_per_shot) * 100,
    name="Win %",
    marker_color="steelblue",
))

# Random baseline
fig.add_hline(
    y=expected_pct,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Random: {expected_pct:.4f}%",
    annotation_position="top right",
)

fig.update_layout(
    title=f"Which Rank Wins Most Often? (Top {n_display} shown)",
    xaxis_title="Error Rank",
    yaxis_title="Win Percentage (%)",
    showlegend=False,
)

fig.show()

## 8. Conditional Analysis by Shot Difficulty

In [ ]:
# ==============================================================================
# Conditional Analysis by Shot Difficulty
# ==============================================================================

print("=" * 60)
print("CONDITIONAL ANALYSIS BY SHOT DIFFICULTY")
print("=" * 60)
print("\nDoes correlation vary with shot difficulty (baseline LLR)?")

if n_shots >= 4:
    # Stratify by best_llr quartiles
    df_per_shot["difficulty_quartile"] = pd.qcut(
        df_per_shot["best_llr"], 
        q=min(4, n_shots),
        labels=["Q1 (easy)", "Q2", "Q3", "Q4 (hard)"][:min(4, n_shots)]
    )

    # Compute statistics per quartile
    quartile_stats = df_per_shot.groupby("difficulty_quartile", observed=True).agg({
        "spearman_rho": ["mean", "std", "count"],
        "rank0_is_best": "mean",
        "regret": "mean",
        "best_llr": ["min", "max"],
    }).round(4)

    quartile_stats.columns = [
        "rho_mean", "rho_std", "n_shots",
        "rank0_success", "mean_regret",
        "llr_min", "llr_max"
    ]

    print("\nPer-quartile statistics:")
    print(quartile_stats.to_string())
else:
    print(f"\nInsufficient shots ({n_shots}) for quartile analysis. Need at least 4 shots.")

## 9. Summary & Conclusions

In [ ]:
print("=" * 60)
print("SUMMARY (EXHAUSTIVE ANALYSIS)")
print("=" * 60)

print(f"\n--- Code & Analysis Parameters ---")
print(f"BB code: [[{metadata.get('n_qubits', N_QUBITS)}, 12, {metadata.get('distance', '?')}]]")
print(f"Physical error rate: {metadata.get('p', P)}")
print(f"Total logical classes: {metadata.get('total_logical_classes', total_logical_classes)}")
print(f"Shots analyzed: {n_shots}")
print(f"Errors per shot: {n_errors_per_shot}")

print(f"\n--- Per-Shot Correlation ---")
print(f"Mean Spearman ρ: {mean_rho:.4f} ± {std_rho:.4f}")
if t_pval is not None:
    print(f"Significantly different from 0: {'Yes' if t_pval < 0.05 else 'No'} (p = {t_pval:.2e})")

print(f"\n--- Rank-0 Selection (most likely error) ---")
print(f"Success rate: {100*rank0_success_rate:.2f}%")
print(f"Random baseline: {100*random_baseline:.4f}%")
print(f"Improvement factor: {rank0_success_rate / random_baseline:.1f}x")

print(f"\n--- Regret ---")
print(f"Mean regret (rank-0): {mean_regret:.2f}")
print(f"Mean regret (random): {mean_random_regret:.2f}")
print(f"Regret reduction: {mean_random_regret - mean_regret:.2f}")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)

# Interpret based on results
if abs(mean_rho) < 0.1:
    print("\n• Per-shot correlation is NEGLIGIBLE (|ρ| < 0.1)")
    print("  → Distribution rank does NOT strongly predict LLR ordering within shots")
elif mean_rho > 0.1:
    print(f"\n• Per-shot correlation is POSITIVE (ρ = {mean_rho:.3f})")
    print("  → Higher rank tends to have higher LLR (distribution is useful)")
else:
    print(f"\n• Per-shot correlation is NEGATIVE (ρ = {mean_rho:.3f})")
    print("  → Higher rank tends to have LOWER LLR (unexpected)")

if rank0_success_rate > random_baseline * 2:
    print(f"\n• Rank-0 selection has STRONG advantage ({rank0_success_rate/random_baseline:.1f}x random)")
elif rank0_success_rate > random_baseline * 1.2:
    print(f"\n• Rank-0 selection has MODERATE advantage ({rank0_success_rate/random_baseline:.1f}x random)")
else:
    print(f"\n• Rank-0 selection has MINIMAL/NO advantage ({rank0_success_rate/random_baseline:.1f}x random)")